In [13]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "tdlib").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find the project root")

In [24]:
import importlib
import src.tdlib.client as tdlib_client
from config.settings import settings

importlib.reload(tdlib_client)
TdlibClient = tdlib_client.TdlibClient

In [23]:
client = TdlibClient(
    api_id=settings.telegram_api_id,
    api_hash=settings.telegram_api_hash,
    phone_number=settings.telegram_phone_number,
    database_directory=settings.tdlib_database_dir,
)

groups = []

await client.start_async()
try:
    group_ids = settings.telegram_group_ids or await client.joined_group_ids_async()
    for group_id in group_ids:
        group_title = await client.get_group_title_async(group_id)
        groups.append({
            "group_id": group_id,
            "group_title": group_title
        })
finally:
    await client.close_async()

/Users/yusufaxmad/Documents/ai/Job-Posting-Analysis/.venv/lib/python3.14/site-packages/telethon/client/auth.py:155: UserWarning: the session already had an authorized user so it did not login to the user account using the provided phone; if you were expecting a different user, check whether you are accidentally reusing an existing session
  warnings.warn(


In [25]:
import pandas as pd

gr_df = pd.DataFrame(groups)
gr_df.to_csv("../../data/telegram_groups.csv")